# Phase 0.5 - Observability Estimators

Notebook này chạy bước tiếp theo sau Gate 2.5 và Phase 0.5 preflight: feature extraction + task-contrast observability estimator workflow.

Phạm vi khoa học:
- Chỉ chạy Phase 0.5 observability estimators.
- Không train decoder Phase 1.
- Không tạo claim privileged-transfer efficacy.
- Không được xem teacher survival draft là kết luận cuối nếu spatial/ICA controls hoặc permutation count final chưa hoàn tất.

Source of truth đã khóa:
- Gate 0: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/20260417T102811097110Z`
- Gate 1: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate1/20260418T153918409528Z`
- Gate 2: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate2/20260418T160143330194Z`
- Gate 2.5: `/content/drive/MyDrive/eeg-ds004752/artifacts/prereg/20260418T161442014597Z`
- Phase 0.5 preflight: `/content/drive/MyDrive/eeg-ds004752/artifacts/phase05/20260418T163438037205Z`


In [ ]:
# ============================================================
# Cell 1: Mount Drive, clone/pull repo, install signal runtime
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Đảm bảo repo private đã clone tại /content/eeg-ds004752.
# - Pull code mới nhất có phase05_estimators CLI.
# - Cài signal extras: mne, numpy, scipy, scikit-learn nếu bootstrap hỗ trợ.
#
# Token GitHub không được in ra output.
# ============================================================

from pathlib import Path
from google.colab import drive
import base64
import getpass
import os
import subprocess

# Mount Drive.
drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True, capture_output=False):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=capture_output)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private nếu cần: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

%cd /content/eeg-ds004752

pull = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull.returncode != 0:
    header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={header}', 'pull'], cwd=REPO_DIR)

# Signal extras cần cho EDF reading và feature extraction.
run(['bash', '-lc', 'INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 2: Khai báo source of truth và đường dẫn output
# ============================================================
# Không dùng latest.txt cho các gate đã khóa, vì source of truth phải cố định.
# ============================================================

from pathlib import Path
import json
import hashlib

REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'

GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260417T102811097110Z'
GATE1_RUN = DRIVE_ROOT / 'artifacts/gate1/20260418T153918409528Z'
GATE2_RUN = DRIVE_ROOT / 'artifacts/gate2/20260418T160143330194Z'
PREREG_RUN = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'
PHASE05_PREFLIGHT_RUN = DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'

PREREG_BUNDLE = PREREG_RUN / 'prereg_bundle.json'
PHASE05_OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase05_estimators'

required_paths = {
    'repo': REPO_DIR,
    'dataset_root': DATASET_ROOT,
    'gate0_run': GATE0_RUN,
    'gate1_run': GATE1_RUN,
    'gate2_run': GATE2_RUN,
    'prereg_run': PREREG_RUN,
    'phase05_preflight_run': PHASE05_PREFLIGHT_RUN,
    'prereg_bundle': PREREG_BUNDLE,
}

print('================ Required Paths ================')
for name, path in required_paths.items():
    print(name, path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {name} -> {path}')

print('Phase 0.5 estimator output root:', PHASE05_OUTPUT_ROOT)


In [ ]:
# ============================================================
# Cell 3: Verify prereg bundle, Phase 0.5 preflight, hash chain
# ============================================================
# Mục tiêu:
# - Kiểm tra prereg bundle locked.
# - Kiểm tra Phase 0.5 preflight bám đúng prereg hash.
# - Kiểm tra bundle trỏ đúng Gate 0/1/2 source of truth.
# ============================================================

import json
from pathlib import Path

bundle = json.loads(PREREG_BUNDLE.read_text())
phase05_summary = json.loads((PHASE05_PREFLIGHT_RUN / 'phase05_summary.json').read_text())
phase05_inputs = json.loads((PHASE05_PREFLIGHT_RUN / 'phase05_inputs.json').read_text())

print('================ Source Integrity ================')
print('Bundle status:', bundle['status'])
print('Bundle hash:', bundle['prereg_bundle_hash_sha256'])
print('Phase 0.5 status:', phase05_summary['status'])
print('Phase 0.5 hash:', phase05_summary['prereg_bundle_hash_sha256'])
print('Gate 0 in bundle:', bundle['source_runs']['gate0'])
print('Gate 1 in bundle:', bundle['source_runs']['gate1'])
print('Gate 2 in bundle:', bundle['source_runs']['gate2'])

assert bundle['status'] == 'locked'
assert Path(bundle['source_runs']['gate0']) == GATE0_RUN
assert Path(bundle['source_runs']['gate1']) == GATE1_RUN
assert Path(bundle['source_runs']['gate2']) == GATE2_RUN
assert phase05_summary['status'] == 'phase05_observability_preflight_ready'
assert phase05_summary['prereg_bundle_hash_sha256'] == bundle['prereg_bundle_hash_sha256']
assert phase05_inputs['prereg_bundle_hash_sha256'] == bundle['prereg_bundle_hash_sha256']

print('OK: Source-of-truth chain is consistent.')


In [ ]:
# ============================================================
# Cell 4: Ghi rõ execution rules dùng cho notebook này
# ============================================================
# Các rule này được đối chiếu từ Technical Spec / Annex / Dossier:
# - Phase 0.5 chỉ observability-only.
# - Mọi fit observability phải nằm trong outer-train.
# - Teacher observable cần task contrast và controls.
# - Spatial/ICA controls nếu chưa chạy thì phải là blocker, không được giả lập thành pass.
# ============================================================

execution_rules = {
    'phase': 'phase05_observability_estimators',
    'decoder_training_allowed': False,
    'privileged_transfer_claim_allowed': False,
    'outer_test_fit_allowed': False,
    'required_observability_quantities': ['Q2_task', 'Q2_base', 'DeltaQ2_obs', 'p_perm'],
    'required_controls': [
        'task_vs_matched_control',
        'grouped_teacher_permutation',
        'nuisance_only_control',
        'spatial_permutation_control',
        'ica_robustness_control',
    ],
    'integrity_policy': 'If a required control is not implemented, report it as blocker; do not mark teacher survival final.',
}

print(json.dumps(execution_rules, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# Cell 5: Run tests and verify CLI availability
# ============================================================
# Nếu tests fail thì dừng. Không chạy estimator trên dữ liệu thật khi code chưa qua test.
# ============================================================

import subprocess

print('\n$ python -m unittest discover -s tests')
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=True)

print('\n$ python -m src.cli phase05_estimators --help')
subprocess.run(['python', '-m', 'src.cli', 'phase05_estimators', '--help'], cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 6: Data materialization sanity check
# ============================================================
# Kiểm tra một vài EDF/iEEG payload đã materialized thật.
# Không tải lại dữ liệu ở đây; nếu thiếu payload thì quay lại DataLad materialization.
# ============================================================

sample_paths = [
    DATASET_ROOT / 'sub-01/ses-01/eeg/sub-01_ses-01_task-verbalWM_run-01_eeg.edf',
    DATASET_ROOT / 'sub-01/ses-01/ieeg/sub-01_ses-01_task-verbalWM_run-01_ieeg.edf',
    DATASET_ROOT / 'sub-15/ses-04/eeg/sub-15_ses-04_task-verbalWM_run-01_eeg.edf',
    DATASET_ROOT / 'sub-15/ses-04/ieeg/sub-15_ses-04_task-verbalWM_run-01_ieeg.edf',
]

print('================ Payload Sanity Check ================')
for path in sample_paths:
    target = path.resolve()
    exists = target.exists()
    size_mb = round(target.stat().st_size / 1024 / 1024, 2) if exists else None
    print('link:', path)
    print('  target_exists:', exists)
    print('  target_size_mb:', size_mb)
    if not exists:
        raise FileNotFoundError(f'Missing materialized payload target: {path}')


In [ ]:
# ============================================================
# Cell 7: Run Phase 0.5 estimator smoke run
# ============================================================
# Smoke run mặc định:
# - 3 subjects đầu tiên.
# - 6 sessions đầu tiên.
# - 24 clean trials/session.
# - 20 grouped permutations.
#
# Đây là implementation/compute smoke, không phải final inference.
# Final inference tối thiểu phải tăng permutations theo registry/config và chạy full cohort.
# ============================================================

SMOKE_MAX_SUBJECTS = 3
SMOKE_MAX_SESSIONS = 6
SMOKE_MAX_TRIALS_PER_SESSION = 24
SMOKE_N_PERMUTATIONS = 20

cmd = [
    'python', '-m', 'src.cli', 'phase05_estimators',
    '--profile', 't4_safe',
    '--prereg-bundle', str(PREREG_BUNDLE),
    '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--config', 'configs/phase05/estimators.json',
    '--output-root', str(PHASE05_OUTPUT_ROOT),
    '--max-subjects', str(SMOKE_MAX_SUBJECTS),
    '--max-sessions', str(SMOKE_MAX_SESSIONS),
    '--max-trials-per-session', str(SMOKE_MAX_TRIALS_PER_SESSION),
    '--n-permutations', str(SMOKE_N_PERMUTATIONS),
]

print('\n$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 8: Inspect smoke run artifacts
# ============================================================
# Mục tiêu:
# - Kiểm tra artifact bắt buộc.
# - In summary ngắn.
# - Xác nhận claim_ready = False.
# ============================================================

LATEST_ESTIMATORS = PHASE05_OUTPUT_ROOT / 'latest.txt'
if not LATEST_ESTIMATORS.exists():
    raise FileNotFoundError(f'Missing latest estimator pointer: {LATEST_ESTIMATORS}')

estimator_run = Path(LATEST_ESTIMATORS.read_text().strip())
summary_path = estimator_run / 'phase05_estimators_summary.json'
summary = json.loads(summary_path.read_text())

required_files = [
    'phase05_estimator_inputs.json',
    'feature_extraction_report.json',
    'task_contrast_observability_results.json',
    'controls_report.json',
    'teacher_survival_table.json',
    'coverage_registry.json',
    'phase05_estimators_report.md',
    'phase05_estimators_summary.json',
]

print('================ Phase 0.5 Estimator Smoke Summary ================')
print('Run dir:', estimator_run)
print('Status:', summary['status'])
print('Subjects:', summary['n_subjects'])
print('Sessions:', summary['n_sessions'])
print('Trials:', summary['n_trials'])
print('Teacher targets:', summary['n_teacher_targets'])
print('Permutations:', summary['n_permutations'])
print('Permutation inference:', summary['permutation_inference_status'])
print('Claim ready:', summary['claim_ready'])
print('Controls blockers:', summary['controls_blockers'])
print('Next step:', summary['next_step'])

assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True
assert summary['claim_ready'] is False

print('\n================ Artifact Check ================')
for name in required_files:
    path = estimator_run / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing estimator artifact: {path}')

print('\nOK: Smoke estimator run completed with explicit non-claim status.')


In [ ]:
# ============================================================
# Cell 9: Optional full-cohort estimator run
# ============================================================
# Mặc định tắt để tránh chạy lâu ngoài ý muốn.
# Chỉ bật RUN_FULL_ESTIMATORS = True khi:
# - Drive còn đủ dung lượng.
# - Colab runtime ổn định.
# - Bạn chấp nhận thời gian chạy dài hơn.
#
# Dù full run xong, nếu spatial/ICA controls còn blocker thì vẫn chưa claim-ready.
# ============================================================

RUN_FULL_ESTIMATORS = False

FULL_MAX_SUBJECTS = 15
FULL_MAX_SESSIONS = 68
FULL_MAX_TRIALS_PER_SESSION = 0  # 0 = dùng toàn bộ clean trials được chọn trong CLI.
FULL_N_PERMUTATIONS = 200

if RUN_FULL_ESTIMATORS:
    full_cmd = [
        'python', '-m', 'src.cli', 'phase05_estimators',
        '--profile', 't4_safe',
        '--prereg-bundle', str(PREREG_BUNDLE),
        '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--config', 'configs/phase05/estimators.json',
        '--output-root', str(PHASE05_OUTPUT_ROOT),
        '--max-subjects', str(FULL_MAX_SUBJECTS),
        '--max-sessions', str(FULL_MAX_SESSIONS),
        '--max-trials-per-session', str(FULL_MAX_TRIALS_PER_SESSION),
        '--n-permutations', str(FULL_N_PERMUTATIONS),
    ]
    print('\n$', ' '.join(full_cmd))
    subprocess.run(full_cmd, cwd=REPO_DIR, check=True)
else:
    print('Full estimator run is disabled. Set RUN_FULL_ESTIMATORS = True to run full cohort.')


In [ ]:
# ============================================================
# Cell 10: Register estimator source of truth pointer
# ============================================================
# Ghi lại run hiện tại để bước sau không dùng latest.txt mơ hồ.
# Nếu bạn đã chạy full run ở Cell 9, pointer sẽ trỏ full run mới nhất.
# Nếu chưa, pointer trỏ smoke run và phải được ghi rõ là smoke/non-claim.
# ============================================================

from datetime import datetime, timezone

estimator_run = Path((PHASE05_OUTPUT_ROOT / 'latest.txt').read_text().strip())
summary = json.loads((estimator_run / 'phase05_estimators_summary.json').read_text())
controls = json.loads((estimator_run / 'controls_report.json').read_text())

source_record = {
    'status': 'phase05_estimators_source_recorded',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'estimator_run': str(estimator_run),
    'summary_path': str(estimator_run / 'phase05_estimators_summary.json'),
    'phase05_preflight_source_of_truth': str(PHASE05_PREFLIGHT_RUN),
    'prereg_bundle': str(PREREG_BUNDLE),
    'prereg_bundle_hash_sha256': bundle['prereg_bundle_hash_sha256'],
    'run_status': summary['status'],
    'claim_ready': summary['claim_ready'],
    'controls_blockers': controls['blockers'],
    'scientific_integrity_limits': summary['scientific_integrity_limits'],
}

source_path = estimator_run / 'phase05_estimators_source_of_truth.json'
source_path.write_text(json.dumps(source_record, indent=2, ensure_ascii=False) + '\n')

print('================ Source Record ================')
print('Written:', source_path)
print('Estimator source:', estimator_run)
print('Run status:', summary['status'])
print('Claim ready:', summary['claim_ready'])
print('Controls blockers:', controls['blockers'])


In [ ]:
# ============================================================
# Cell 11: Final interpretation
# ============================================================
# Cell này không thêm tính toán. Chỉ in diễn giải đúng giới hạn.
# ============================================================

report_text = (estimator_run / 'phase05_estimators_report.md').read_text()
print('================ Report Preview ================')
print(report_text[:3000])

print('\n================ Final Interpretation ================')
print('OK: Phase 0.5 estimator workflow ran under the locked prereg bundle and fixed Phase 0.5 preflight source.')
print('OK: Feature extraction, task/base contrast, grouped teacher permutation, and nuisance-only control outputs were written.')
print('NOT OK TO CLAIM: This is not decoder training and not privileged-transfer efficacy evidence.')
print('NOT FINAL IF BLOCKERS EXIST: Teacher survival remains draft while spatial/ICA controls or final permutation count are incomplete.')
print('NEXT: Implement spatial permutation and ICA robustness controls, then rerun with final permutation count before any teacher-survival claim.')
print('Phase 0.5 estimator source of truth:', estimator_run)
